### This is about query expension - We will learn about query expension

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableMap
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


### Loading the text from queryexpension-data.txt

In [6]:
loader = TextLoader("queryexpension-data.txt", encoding="utf-8")
documents = loader.load()
documents

[Document(metadata={'source': 'queryexpension-data.txt'}, page_content="LangChain and AI Development Notes\n\n## LangChain\n\nLangChain is a framework for building applications powered by large language models (LLMs).\nIt provides reusable components that connect models to prompts, data sources, tools, and application workflows.\n\nA workflow can combine multiple steps.\nFor example, it can accept a question, retrieve relevant documents, provide those documents to an LLM, and return an answer.\n\n## Building an AI Application\n\n### Choose and Configure a Model\n\nChoose an LLM based on the application's quality, speed, privacy, and cost needs.\nA model may be accessed through a hosted API or run locally.\nStore API credentials securely and keep secrets out of source code and prompts.\n\n### Write and Test Prompts\n\nWrite prompts that clearly state the task, provide necessary context, and describe the desired response format.\nTest prompts with typical, ambiguous, and unusual inputs.\

In [8]:
### Split into chunks by RecursiveCharacterTextSplitter

chunker = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=25
)
chunks = chunker.split_documents(documents=documents)
chunks
print(f"Chunks size: {len(chunks)}")

Chunks size: 23


### Embedd the documents to store in vectors

In [13]:
from langchain_community.vectorstores import FAISS
embedder = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = FAISS.from_documents(
    embedding=embedder,
    documents=chunks
)

retriver = vector_store.as_retriever(
    search_type="mmr",
    kwargs={"k":3}
)
retriver



VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1288f0440>, search_type='mmr', search_kwargs={})

In [15]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(model="gpt-4o-mini")
query_expension_prompt = ChatPromptTemplate.from_template(
    """ You are helpful assistent. Expand the following query to improve document retrieval by adding relative anonyms, technical term ans useful context.
    Orginal Query: "{query}
    Expended query:
    """
)

query_expension_chain = query_expension_prompt | llm | StrOutputParser()
query_expension_chain

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template=' You are helpful assistent. Expand the following query to improve document retrieval by adding relative anonyms, technical term ans useful context.\n    Orginal Query: "{query}\n    Expended query:\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structu

In [17]:
query_expension_chain.invoke({"query": "Langchain memory"})

'Expanded Query: "Langchain memory systems, cognitive architectures, memory management in machine learning, knowledge retention technologies, neural memory models, applications of Langchain in artificial intelligence, comparative analysis of memory frameworks, memory retrieval mechanisms in AI, contextual memory in language processing, dynamic memory allocation, associative memory techniques, integration of Langchain with neural networks, efficiency of Langchain memory in NLP (Natural Language Processing), alternatives to Langchain for memory management, semantic memory structures, long-term vs. short-term memory in computational systems." \n\nThis expanded query introduces relevant synonyms, technical terms, and contextual elements related to the original topic, which should improve document retrieval effectiveness.'

### Answer Prompt to get actual answer from LLM

In [19]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below
Context: {context}
Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt=answer_prompt)


In [21]:
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriver.invoke(query_expension_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

rag_pipeline.invoke({"input": "python helpful?"})

"Yes, Python is a helpful programming language, particularly in the context of AI and machine learning development. It is widely used for building applications powered by large language models (LLMs) like LangChain, as it offers various libraries and frameworks that facilitate data processing, model training, and implementation of AI algorithms. Additionally, Python's simplicity and readability make it an excellent choice for developers working on complex applications."